In [1]:
"""
Example: Using AnthropicLLM with multiple tool calls (single file).

Claude will:
1. Call get_weather(city="Paris")
2. Call convert_temperature(celsius=<result>)
3. Produce a final answer
"""
import os

from dotenv import load_dotenv
from langchain_core.tools import BaseTool
from pydantic import BaseModel

from buc_factory.llm.claudeai import AnthropicLLM

load_dotenv()  # Load environment variables from .env file

# ------------------------------------------------------------------
# Tool 1: Get weather
# ------------------------------------------------------------------

class GetWeatherInput(BaseModel):
    city: str


class GetWeatherTool(BaseTool):
    name: str = "get_weather"
    description: str = "Get the current temperature in Celsius for a given city."
    args_schema: type[BaseModel] = GetWeatherInput

    def _run(self, city: str) -> str:
        if city.lower() == "paris":
            return "18"
        return "0"


# ------------------------------------------------------------------
# Tool 2: Convert temperature
# ------------------------------------------------------------------

class ConvertTemperatureInput(BaseModel):
    celsius: float


class ConvertTemperatureTool(BaseTool):
    name: str = "convert_temperature"
    description: str = "Convert a temperature from Celsius to Fahrenheit."
    args_schema: type[BaseModel] = ConvertTemperatureInput

    def _run(self, celsius: float) -> str:
        fahrenheit = (celsius * 9 / 5) + 32
        return str(fahrenheit)


# ------------------------------------------------------------------
# Main execution
# ------------------------------------------------------------------

# Initialize Claude LLM
llm = AnthropicLLM(
    api_key=os.getenv("ANTHROPIC_API_KEY"),
    temperature=0.5,
    max_tokens=512,
)

# Bind tools to the model
llm_with_tools = llm.bind_tools(
    tools=[
        GetWeatherTool(),
        ConvertTemperatureTool(),
    ],
    tool_choice="auto",  # let Claude decide which tool(s) to call
)

# Prompt that requires multiple tools
prompt = (
    "You MUST use the tools to answer the following question: "
    "What is the weather today in Paris, "
    "convert it to Fahrenheit, and explain it briefly."
)

# Invoke the model
result = llm_with_tools.invoke(prompt)

# Print final response
print("\n=== Final Answer ===")
print(result.content)


Model requested 1 tool call(s): ['get_weather']
Executed tool 'get_weather' with args {'city': 'Paris'}, result: 18
Model requested 1 tool call(s): ['convert_temperature']
Executed tool 'convert_temperature' with args {'celsius': 18}, result: 64.4

=== Final Answer ===
Here's a brief summary of today's weather in Paris:

🌤️ **Paris Weather Today**
- **Temperature:** 18°C / **64.4°F**

**Brief Explanation:**
At 18°C (64.4°F), Paris is experiencing a **mild and pleasant day**. This is a comfortable temperature — not too hot and not too cold. It's the kind of weather where you might want to bring a light jacket or sweater, especially in the morning or evening when it could feel slightly cooler. Perfect for a stroll along the Seine or exploring the city! 🗼


In [ ]:
import logging
import os

import anthropic
from dotenv import load_dotenv

load_dotenv()

logging.basicConfig(level=logging.INFO, 
                    format="%(asctime)s - %(levelname)s - %(message)s",
                    handlers=[logging.StreamHandler()])
LOGGER = logging.getLogger(__name__)

client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
master_prompt = "Tell me a story about a brave knight." #Path("prompt_only.txt").read_text(encoding="utf-8")

# Stream the response from the model and print it in real-time
with client.messages.stream(
    model="claude-opus-4-7",
    max_tokens=2000,
    messages=[{"role": "user", "content": master_prompt}],
) as stream:
    full_text = []
    for text in stream.text_stream:
        print(text, end="", flush=True)
        full_text.append(text)

In [1]:
import os

from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()

client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

available_models = client.models.list()
models_list = [model.id for model in available_models.data]

print("Available models:")
for model_id in models_list:
    print(f"- {model_id}")

Available models:
- claude-opus-4-7
- claude-sonnet-4-6
- claude-opus-4-6
- claude-opus-4-5-20251101
- claude-haiku-4-5-20251001
- claude-sonnet-4-5-20250929
- claude-opus-4-1-20250805
- claude-opus-4-20250514
- claude-sonnet-4-20250514


In [ ]:
actual_entities = {'stores', 'products', 'sales_orders', 'inventory_snapshots', 'order_lines', 'customers'}
expected_entities = {'customers.csv', 'order_lines.csv', 'products.csv', 'inventory_snapshots.csv', 'sales_orders.csv', 'stores.csv'}